# BL-YOLOv8-S — CH-RDD2022 (Kaggle)

Notebook ini mereplikasi proposed method **BL-YOLOv8** pada YOLOv8-S: SimSPPF (Conv-BN-ReLU), LSK-Attention setelah backbone, dan weighted BiFPN pada neck. Backbone dan Detect tetap YOLOv8-S.

Gunakan split yang sama, seed, pretrained, dan hyperparameter yang sama ketika membandingkan dengan YOLOv8s baseline. Aktifkan GPU dan Internet pada Kaggle.

In [ ]:
# 1. Clone branch yang berisi implementasi BL-YOLOv8 lalu install source-nya.
import json
import platform
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-bl-replica'
REPO_DIR = Path('/kaggle/working/yolo-aceh-rdd2022')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=True)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('Python          :', platform.python_version())
print('Branch / commit :', REPO_BRANCH, '/', REPO_COMMIT)


In [ ]:
# 2. Data CH-RDD2022 dan hyperparameter Table 3 pada paper.
import sys
import torch
import yaml

sys.path.insert(0, str(REPO_DIR))
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split-8-1-1')
# Ganti hanya DATA_ROOT jika nama Kaggle Dataset Anda berbeda; harus menunjuk folder split 8:1:1.
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset split 8:1:1 tidak ditemukan: {DATA_ROOT}')

NC = 5
NAMES = {0: 'D00', 1: 'D10', 2: 'D20', 3: 'D40', 4: 'Repair'}
DATA_YAML = Path('/kaggle/working/ch_rdd2022_8_1_1.yaml')
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(DATA_ROOT), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
    'nc': NC, 'names': NAMES,
}, sort_keys=False), encoding='utf-8')

MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/v8/bl-yolov8s.yaml'
EPOCHS, IMGSZ, BATCH = 160, 640, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
SEED, PATIENCE, DEVICE = 42, 0, 0

extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
split_counts = {split: sum(p.suffix.lower() in extensions for p in (DATA_ROOT / split / 'images').glob('*'))
                for split in ('train', 'val', 'test')}
total_images = sum(split_counts.values())
ratios = {key: round(value / total_images, 4) for key, value in split_counts.items()}
print('DATASET CHECK')
print('Root   :', DATA_ROOT)
print('Counts :', split_counts, 'total=', total_images)
print('Ratios :', ratios)
if total_images != 4373:
    print('WARNING: paper memakai 4,373 citra setelah filtering; laporkan perbedaan ini.')
print('Table 3 settings:', {'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'optimizer': OPTIMIZER,
                           'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED})


In [ ]:
# 3. Print model dan cek komponen proposed method sebelum training.
from ultralytics.nn.modules import BiFPNFusion, LSKAttention, SimSPPF
from ultralytics.nn.tasks import DetectionModel

baseline = DetectionModel('yolov8s.yaml', nc=NC, verbose=False)
bl_model = DetectionModel(str(MODEL_YAML), nc=NC, verbose=True)
special_layers = [(layer.i, layer.__class__.__name__, getattr(layer, 'weights', None).numel()
                   if hasattr(layer, 'weights') else None) for layer in bl_model.model
                  if isinstance(layer, (SimSPPF, LSKAttention, BiFPNFusion))]
assert [item[1] for item in special_layers] == ['SimSPPF', 'LSKAttention', 'BiFPNFusion', 'BiFPNFusion',
                                                'BiFPNFusion', 'BiFPNFusion', 'BiFPNFusion']
assert [item[2] for item in special_layers if item[1] == 'BiFPNFusion'] == [2, 2, 3, 3, 2]
bl_model.eval()
with torch.inference_mode():
    output = bl_model(torch.zeros(1, 3, IMGSZ, IMGSZ))
assert isinstance(output, tuple) and output[0].shape[1] == NC + 4
assert len(bl_model.model[-1].stride) == 3
print('MODEL CHECK PASSED')
print('YOLOv8s parameters   :', f'{sum(p.numel() for p in baseline.parameters()):,}')
print('BL-YOLOv8s parameters:', f'{sum(p.numel() for p in bl_model.parameters()):,}')
print('Proposed layers      :', special_layers)
del baseline, bl_model, output
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 4. Mulai dari yolov8s.pt; transfer hanya backbone YOLOv8 yang struktur dan bentuk tensor-nya tetap identik (0--8).
import re
from ultralytics import YOLO

PRETRAINED_WEIGHTS = 'yolov8s.pt'
model = YOLO(str(MODEL_YAML), task='detect')
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
source_state, target_state = source_model.state_dict(), model.model.state_dict()
pattern = re.compile(r'^model\.(\d+)\..+$')
transferred = {key: tensor for key, tensor in source_state.items()
               if (match := pattern.match(key)) and int(match.group(1)) <= 8
               and key in target_state and target_state[key].shape == tensor.shape}
incompatible = model.model.load_state_dict(transferred, strict=False)
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'policy': 'only matching YOLOv8 backbone layers 0--8; SimSPPF, LSK, BiFPN, and Detect are fresh',
    'transferred_tensors': len(transferred), 'target_tensors': len(target_state),
    'uninitialized_tensors': len(incompatible.missing_keys),
}
# Trainer akan membangun head 5 kelas dan memuat seluruh tensor yang kompatibel dari checkpoint ini.
model.ckpt = {'model': model.model}
print('PRETRAINED WEIGHT TRANSFER')
print(json.dumps(PRETRAINED_REPORT, indent=2))
del source_model, source_state, target_state, transferred
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 5. Training paper setting. Jika BATCH=64 tidak muat di GPU, ubah dan catat sebagai eksperimen non-identik.
RUN_PROJECT = '/kaggle/working/bl_yolov8_runs'
RUN_NAME = 'bl_yolov8s_ch_rdd2022'
print('TRAINING START')
train_results = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    optimizer=OPTIMIZER, lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    seed=SEED, deterministic=True, patience=PATIENCE, pretrained=True,
    project=RUN_PROJECT, name=RUN_NAME, exist_ok=True, workers=4, plots=True, verbose=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
assert BEST_PT.is_file(), f'best checkpoint tidak ditemukan: {BEST_PT}'
print('TRAINING FINISHED:', RUN_DIR)
print('BEST CHECKPOINT  :', BEST_PT)


In [ ]:
# 6. Evaluasi validation/test dan buat ZIP hasil yang bisa diunduh dari Kaggle Output.
from zipfile import ZIP_DEFLATED, ZipFile

best_model = YOLO(str(BEST_PT))
def metrics_dict(metrics):
    return {'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
            'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map)}

val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=16, device=DEVICE,
                             project=str(RUN_DIR), name='val_final', exist_ok=True, plots=True)
test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=16, device=DEVICE,
                              project=str(RUN_DIR), name='test_final', exist_ok=True, plots=True)
FINAL_METRICS = {'validation': metrics_dict(val_metrics), 'test': metrics_dict(test_metrics)}
METRICS_JSON = RUN_DIR / 'evaluation_metrics.json'
METRICS_JSON.write_text(json.dumps({
    'metrics': FINAL_METRICS, 'dataset_counts': split_counts, 'dataset_total': total_images,
    'repository_url': REPO_URL, 'repository_branch': REPO_BRANCH, 'repository_commit': REPO_COMMIT,
    'model_yaml': str(MODEL_YAML), 'pretrained_transfer': PRETRAINED_REPORT,
    'method': 'BL-YOLOv8-S: SimSPPF + LSK-Attention + weighted BiFPN',
    'hyperparameters': {'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'optimizer': OPTIMIZER,
                        'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED},
}, indent=2), encoding='utf-8')
print('FINAL METRICS')
print(json.dumps(FINAL_METRICS, indent=2))

ZIP_PATH = Path('/kaggle/working/bl_yolov8s_ch_rdd2022_results.zip')
source_files = [REPO_DIR / 'ultralytics/nn/modules/block.py', REPO_DIR / 'ultralytics/nn/modules/__init__.py',
                REPO_DIR / 'ultralytics/nn/tasks.py', MODEL_YAML]
with ZipFile(ZIP_PATH, 'w', ZIP_DEFLATED) as archive:
    for file in RUN_DIR.rglob('*'):
        if file.is_file():
            archive.write(file, Path('run') / file.relative_to(RUN_DIR))
    archive.write(DATA_YAML, 'configuration/ch_rdd2022_8_1_1.yaml')
    for file in source_files:
        archive.write(file, Path('source') / file.relative_to(REPO_DIR))
print('ZIP READY:', ZIP_PATH, f'({ZIP_PATH.stat().st_size / 1024**2:.2f} MB)')
